# XGBoost — Network Intrusion Detection

Huấn luyện và đánh giá mô hình **XGBoost** trên tập dữ liệu CICIDS2017.
XGBoost yêu cầu label encoding (nhãn dạng số) và sử dụng sample weights để xử lý mất cân bằng.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))
sys.path.append(os.path.abspath(".."))

from src.model_training import (
    load_splits,
    evaluate_model,
    plot_confusion_matrix,
    compare_models,
)

In [ ]:
# --- STEP 1: Load dữ liệu đã chia sẵn ---
print("=" * 70)
print("STEP 1: Loading pre-split data")
print("=" * 70)

X_train, X_test, y_train, y_test = load_splits()

print(f"Feature columns: {X_train.shape[1]}")
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Classes: {sorted(y_train.unique())}")

In [ ]:
# --- STEP 2: Label encoding + sample weights ---
print("\n" + "=" * 70)
print("STEP 2: Label Encoding & Sample Weights")
print("=" * 70)

# XGBoost cần nhãn dạng số (0, 1, 2, ...)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Tính sample weights để xử lý class imbalance
weights = compute_sample_weight(class_weight="balanced", y=y_train_encoded)

print(f"Label encoding applied — {len(le.classes_)} classes")
print(f"Sample weights computed — shape: {weights.shape}")

In [ ]:
# --- STEP 3: Train XGBoost ---
print("\n" + "=" * 70)
print("STEP 3: Training XGBoost (n_estimators=300)")
print("=" * 70)

t0 = time.time()

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=1,
    eval_metric="mlogloss",
)

print("Training model with sample weights...")
xgb_model.fit(X_train, y_train_encoded, sample_weight=weights)

train_time = time.time() - t0
print(f"\nTraining completed in {train_time:.2f} seconds")

In [ ]:
# --- STEP 4: Evaluate ---
print("\n" + "=" * 70)
print("STEP 4: Model Evaluation")
print("=" * 70)

y_pred_encoded = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)

# Giải mã nhãn số về chuỗi gốc
y_pred = le.inverse_transform(y_pred_encoded)

xgb_results = evaluate_model(
    y_true=y_test,
    y_pred=y_pred,
    model_name="XGBoost (Balanced, n_estimators=300)",
    y_pred_proba=y_pred_proba,
    labels=le.classes_.tolist(),
)

In [ ]:
# --- STEP 5: Confusion Matrix ---
print("\n" + "=" * 70)
print("STEP 5: Confusion Matrix")
print("=" * 70)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    labels=le.classes_.tolist(),
    model_name="XGBoost (Balanced)",
    normalize=True,
    figsize=(12, 10),
)

In [ ]:
# --- STEP 6: Feature Importance ---
print("\n" + "=" * 70)
print("STEP 6: Feature Importance (Top 20)")
print("=" * 70)

feat_imp = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print(feat_imp.head(20).to_string(index=False))

plt.figure(figsize=(12, 6))
sns.barplot(
    data=feat_imp.head(20),
    x="Importance", y="Feature",
    hue="Feature", legend=False,
    palette="viridis",
)
plt.title("Top 20 Feature Importance — XGBoost")
plt.xlabel("Importance Score")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 7: Summary ---
print("\n" + "=" * 70)
print("STEP 7: Summary")
print("=" * 70)

comparison_df = compare_models([xgb_results])
print("\nPerformance Metrics:")
print(comparison_df.to_string(index=False))

print(f"\nTraining time  : {train_time:.2f}s")
print(f"Test set size  : {len(y_test):,}")
print(f"Estimators     : 300")
print(f"Features       : {X_train.shape[1]}")
print(f"Classes        : {len(le.classes_)}")